Əvvəl notebooku və SQL bağlantısını hazırlayırıq.

In [19]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

# Layihənin əsas qovluğunu tapırıq
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

from db import get_engine

# SQL Server bağlantısı
engine = get_engine()

print("Silver mərhələsi üçün Python və SQL Server hazırdır!")

Silver mərhələsi üçün Python və SQL Server hazırdır!


Kredit məlumatlarını oxuyaq

In [20]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
with engine.connect() as conn:
    silver_loans_df = pd.read_sql(
        text("SELECT * FROM bronze.loans_enriched"),
        conn
    )

In [21]:
silver_loans_df.head()

,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,T?sdiql?ndi,NaN,0.0,...,0.0,2026-01-01,1,Yeni il,0,0,0,5.5,0.0,18.6
1,L000002,C00082,1/2/2026,Istehlak krediti,14942.94,15,13.74,R?dd edildi,Yüks?k borc yükü,NaN,...,1.0,2026-01-02,1,Yeni il,0,0,0,5.2,12.9,22.6
2,L000003,C00715,1/2/2026,Istehlak krediti,24217.34,58,14.09,T?sdiql?ndi,NaN,0.0,...,1.0,2026-01-02,1,Yeni il,0,0,0,2.9,5.3,36.6
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,R?dd edildi,Asagi kredit skoru,NaN,...,1.0,2026-01-02,1,Yeni il,0,0,0,2.0,0.7,25.5
4,L000005,C00178,1/3/2026,Istehlak krediti,6382.58,21,13.19,T?sdiql?ndi,NaN,0.0,...,0.0,2026-01-03,0,NaN,1,0,0,3.6,0.0,10.7


In [22]:
silver_loans_df.shape

(4820, 31)

In [23]:
silver_loans_df.duplicated().sum()

np.int64(15)

In [27]:
silver_loans_df = silver_loans_df.drop_duplicates()

In [28]:
silver_loans_df.duplicated().sum()

np.int64(0)

In [29]:
silver_loans_df.isna().sum()

loan_id                   0
customer_id               0
application_date          0
product_type              0
amount                    0
term_months               0
interest_rate            50
status                    0
rejection_reason       3070
defaulted              1826
age                       5
gender                    5
city                      5
branch                    5
tenure_years              5
credit_score              5
balance                   5
salary                    5
products_count            5
has_card                  5
is_active                 5
churned                   5
merge_date                0
is_holiday                0
holiday_name           4768
is_weekend                0
is_workday                0
is_pre_holiday            0
temperature_2m_mean       5
precipitation_sum         5
wind_speed_10m_max        5
dtype: int64

In [32]:
silver_loans_df.groupby("status")["defaulted"].apply(lambda x: x.isna().sum())

status
R?dd edildi    1777
T?sdiql?ndi       0
r?dd edildi      49
t?sdiql?ndi       0
Name: defaulted, dtype: int64

defaulted kredit gecikmesini gosterir. ve 1826 bos deyerin hamisi redd edilen kreditlere aiddir. bu normaldir. 

Status yazılışlarını eyniləşdirək

In [33]:
silver_loans_df["status"] = silver_loans_df["status"].str.lower()

In [34]:
silver_loans_df["status"].value_counts()

status
t?sdiql?ndi    2979
r?dd edildi    1826
Name: count, dtype: int64

In [35]:
pd.read_csv(PROJECT_ROOT / "data/bank_data/loans.csv")["status"].unique()

<ArrowStringArray>
['Təsdiqləndi', 'Rədd edildi', 'rədd edildi', 'təsdiqləndi']
Length: 4, dtype: str

In [36]:
pd.read_csv(PROJECT_ROOT / "data/bank_data/customers.csv")[["city", "branch"]].head()

,city,branch
0,Bakı,Yasamal
1,Sumqayıt,Sumqayıt Mərkəz
2,Gəncə,Gəncə Mərkəz
3,Bakı,Səbail
4,Bakı,Nərimanov


In [37]:
pd.read_sql("SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA='bronze' AND TABLE_NAME='loans' AND COLUMN_NAME='status'", engine)

,COLUMN_NAME,DATA_TYPE
0,status,varchar
